# 태양 코로나 및 태양풍의 운동론적 물리학 — 구현\n# Kinetic Physics of the Solar Corona and Solar Wind — Implementation\n\n**논문 / Paper**: Marsch, E. (2006), \"Kinetic Physics of the Solar Corona and Solar Wind\", *Living Rev. Solar Phys.*, 3, 1\n\n**목적 / Purpose**: 이 노트북은 Marsch (2006) 리뷰 논문의 핵심 방정식과 물리적 개념을 Python으로 구현합니다. 속도 분포 함수(VDF), 충돌성, exospheric 모델, 분산 관계, 파동-입자 상호작용, 온도 비등방성 조절 등을 다룹니다.\n\nThis notebook implements key equations and physical concepts from the Marsch (2006) review paper in Python. Topics include velocity distribution functions (VDFs), collisionality, exospheric models, dispersion relations, wave-particle interactions, and temperature anisotropy regulation.\n\n---\n\n## 목차 / Table of Contents\n\n1. **속도 분포 함수 — Maxwellian vs Kappa / Velocity Distribution Functions — Maxwellian vs Kappa**\n2. **태양풍 전자 VDF — Core-Halo-Strahl 모델 / Solar Wind Electron VDF — Core-Halo-Strahl Model**\n3. **태양 대기의 충돌성 / Collisionality in the Solar Atmosphere**\n4. **Exospheric 태양풍 모델 / Exospheric Solar Wind Model**\n5. **분산 관계 — Alfven-Cyclotron 파동 / Dispersion Relations — Alfven-Cyclotron Waves**\n6. **Cyclotron 공명과 Pitch-Angle 확산 / Cyclotron Resonance and Pitch-Angle Diffusion**\n7. **온도 비등방성 조절 / Temperature Anisotropy Regulation**\n8. **현대적 맥락 — Marsch에서 Parker Solar Probe까지 / Modern Context — From Marsch to Parker Solar Probe**

In [ ]:
# Common imports and constants used throughout the notebook.\nimport numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib import cm\nfrom scipy import constants as const\nfrom scipy.special import gamma as gamma_func\nfrom scipy.optimize import brentq\n\n# Physical constants (SI).\nk_B = const.k           # Boltzmann constant [J/K]\nm_e = const.m_e         # Electron mass [kg]\nm_p = const.m_p         # Proton mass [kg]\ne_charge = const.e      # Elementary charge [C]\nc_light = const.c       # Speed of light [m/s]\nepsilon_0 = const.epsilon_0  # Vacuum permittivity [F/m]\nmu_0 = const.mu_0       # Vacuum permeability [H/m]\nG_grav = const.G        # Gravitational constant [m^3 kg^-1 s^-2]\n\n# Solar constants.\nM_sun = 1.989e30        # Solar mass [kg]\nR_sun = 6.957e8         # Solar radius [m]\nAU = 1.496e11           # 1 AU [m]\n\n# Plotting defaults.\nplt.rcParams.update({\n    'figure.figsize': (10, 7),\n    'font.size': 12,\n    'axes.labelsize': 14,\n    'axes.titlesize': 14,\n    'legend.fontsize': 11,\n    'lines.linewidth': 2,\n    'figure.dpi': 100,\n})\n\nprint(\"All imports and constants loaded successfully.\")\nprint(f\"k_B = {k_B:.4e} J/K\")\nprint(f\"m_p = {m_p:.4e} kg\")\nprint(f\"m_e = {m_e:.4e} kg\")\nprint(f\"e   = {e_charge:.4e} C\")\nprint(f\"Solar escape speed = {np.sqrt(2*G_grav*M_sun/R_sun)/1e3:.1f} km/s\")

---\n\n## Part 1: 속도 분포 함수 — Maxwellian vs Kappa\n## Part 1: Velocity Distribution Functions — Maxwellian vs Kappa\n\n### 이론적 배경 / Theoretical Background\n\n열평형 상태의 입자 집합은 **Maxwell 분포**를 따릅니다:\n\nA particle population in thermal equilibrium follows the **Maxwellian distribution**:\n\n$$f_M(v) = \\frac{n}{(\\pi v_{th}^2)^{3/2}} \\exp\\left(-\\frac{v^2}{v_{th}^2}\\right), \\quad v_{th} = \\sqrt{\\frac{2k_BT}{m}}$$\n\n태양풍에서 관측되는 VDF는 고에너지 꼬리(suprathermal tail)를 가지며, 이를 **kappa ($\\kappa$) 분포**로 잘 기술할 수 있습니다 (논문 Eq. 6):\n\nVDFs observed in the solar wind have suprathermal tails, well described by the **kappa ($\\kappa$) distribution** (paper Eq. 6):\n\n$$f_\\kappa(v) = \\frac{n}{(\\pi \\kappa v_\\kappa^2)^{3/2}} \\frac{\\Gamma(\\kappa+1)}{\\Gamma(\\kappa-1/2)} \\left[1 + \\frac{v^2}{\\kappa v_\\kappa^2}\\right]^{-(\\kappa+1)}$$\n\n여기서 $v_\\kappa = \\sqrt{\\frac{2\\kappa - 3}{\\kappa} \\frac{k_BT_\\kappa}{m}}$ 이고, $\\kappa > 3/2$ 이어야 합니다.\n\nwhere $v_\\kappa = \\sqrt{\\frac{2\\kappa - 3}{\\kappa} \\frac{k_BT_\\kappa}{m}}$ and $\\kappa > 3/2$ is required.\n\n**핵심 성질 / Key properties:**\n- $\\kappa \\to \\infty$이면 Maxwell 분포로 수렴 / Converges to Maxwellian as $\\kappa \\to \\infty$\n- 작은 $\\kappa$일수록 고에너지 꼬리가 강함 / Smaller $\\kappa$ means enhanced high-energy tails\n- 고에너지 영역에서 멱법칙 거동: $f(v) \\sim v^{-2(\\kappa+1)}$ / Power-law tail behavior: $f(v) \\sim v^{-2(\\kappa+1)}$

In [ ]:
def maxwellian_1d(v, n, T, m):\n    \"\"\"Compute 1D Maxwellian velocity distribution function.\n\n    Args:\n        v: Velocity array [m/s].\n        n: Number density [m^-3].\n        T: Temperature [K].\n        m: Particle mass [kg].\n\n    Returns:\n        f(v) in units of [s/m] (1D distribution).\n    \"\"\"\n    v_th = np.sqrt(2 * k_B * T / m)\n    return (n / (np.sqrt(np.pi) * v_th)) * np.exp(-(v / v_th)**2)\n\n\ndef maxwellian_3d(v, n, T, m):\n    \"\"\"Compute isotropic 3D Maxwellian VDF evaluated at speed v.\n\n    Args:\n        v: Speed array [m/s].\n        n: Number density [m^-3].\n        T: Temperature [K].\n        m: Particle mass [kg].\n\n    Returns:\n        f(v) in units of [s^3/m^3].\n    \"\"\"\n    v_th = np.sqrt(2 * k_B * T / m)\n    return n / (np.pi * v_th**2)**(1.5) * np.exp(-(v / v_th)**2)\n\n\ndef kappa_3d(v, n, T, m, kappa):\n    \"\"\"Compute isotropic 3D kappa distribution function (Eq. 6 of Marsch 2006).\n\n    Args:\n        v: Speed array [m/s].\n        n: Number density [m^-3].\n        T: Effective temperature [K].\n        m: Particle mass [kg].\n        kappa: Kappa index (must be > 3/2).\n\n    Returns:\n        f(v) in units of [s^3/m^3].\n    \"\"\"\n    if kappa <= 1.5:\n        raise ValueError(\"kappa must be > 3/2 for finite energy.\")\n    v_kappa = np.sqrt((2 * kappa - 3) / kappa * k_B * T / m)\n    prefactor = n / (np.pi * kappa * v_kappa**2)**(1.5)\n    gamma_ratio = gamma_func(kappa + 1) / gamma_func(kappa - 0.5)\n    return prefactor * gamma_ratio * (1 + v**2 / (kappa * v_kappa**2))**(-(kappa + 1))\n\n\ndef kappa_1d(v, n, T, m, kappa):\n    \"\"\"Compute 1D kappa distribution (marginal along one velocity component).\n\n    Args:\n        v: Velocity array [m/s].\n        n: Number density [m^-3].\n        T: Effective temperature [K].\n        m: Particle mass [kg].\n        kappa: Kappa index (must be > 3/2).\n\n    Returns:\n        f(v) in units of [s/m].\n    \"\"\"\n    if kappa <= 1.5:\n        raise ValueError(\"kappa must be > 3/2 for finite energy.\")\n    v_kappa = np.sqrt((2 * kappa - 3) / kappa * k_B * T / m)\n    prefactor = 1.0 / (np.sqrt(np.pi * kappa) * v_kappa)\n    gamma_ratio = gamma_func(kappa + 0.5) / gamma_func(kappa)\n    return n * prefactor * gamma_ratio * (1 + v**2 / (kappa * v_kappa**2))**(-(kappa + 0.5))\n\n\nprint(\"VDF functions defined.\")

### 1.1 Kappa 분포의 Maxwellian 수렴 시각화 / Visualizing Kappa-to-Maxwellian Convergence\n\n아래 그래프는 다양한 $\\kappa$ 값에 대한 kappa 분포를 Maxwell 분포와 비교합니다 (논문 Figure 7 스타일). $\\kappa$가 증가할수록 분포가 Maxwell 분포에 수렴하며, 작은 $\\kappa$에서는 고에너지 꼬리가 뚜렷합니다.\n\nThe plot below compares kappa distributions for various $\\kappa$ values with the Maxwellian (Figure 7 style from the paper). As $\\kappa$ increases, the distribution converges to Maxwellian; small $\\kappa$ shows pronounced high-energy tails.

In [ ]:
# --- Figure 7 style: Maxwellian vs Kappa distributions ---\n\n# Parameters: typical solar wind electrons at 1 AU.\nn = 10e6          # Number density [m^-3] (10 cm^-3)\nT = 1.5e5         # Temperature [K]\nm = m_e           # Electron mass\n\n# Velocity range in units of thermal speed.\nv_th = np.sqrt(2 * k_B * T / m)\nv_norm = np.linspace(0, 8, 1000)  # v / v_th\nv = v_norm * v_th                  # Actual velocity [m/s]\n\n# Compute distributions.\nf_maxwell = maxwellian_3d(v, n, T, m)\nkappa_values = [2, 3, 4, 10]\ncolors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']\n\nfig, axes = plt.subplots(1, 2, figsize=(16, 6))\n\n# --- Left panel: linear scale ---\nax = axes[0]\nax.plot(v_norm, f_maxwell / f_maxwell[1], 'k-', lw=2.5, label='Maxwellian')\nfor kappa, color in zip(kappa_values, colors):\n    f_kap = kappa_3d(v, n, T, m, kappa)\n    ax.plot(v_norm, f_kap / f_maxwell[1], '--', color=color,\n            label=f'$\\\\kappa$ = {kappa}')\nax.set_xlabel('$v / v_{th}$')\nax.set_ylabel('$f(v)$ (normalized / 정규화)')\nax.set_title('선형 스케일 / Linear Scale')\nax.legend()\nax.set_xlim(0, 6)\nax.set_ylim(0, 1.1)\nax.grid(True, alpha=0.3)\n\n# --- Right panel: log scale (shows power-law tails) ---\nax = axes[1]\nax.semilogy(v_norm, f_maxwell / f_maxwell[1], 'k-', lw=2.5, label='Maxwellian')\nfor kappa, color in zip(kappa_values, colors):\n    f_kap = kappa_3d(v, n, T, m, kappa)\n    ax.semilogy(v_norm, f_kap / f_maxwell[1], '--', color=color,\n                label=f'$\\\\kappa$ = {kappa}')\nax.set_xlabel('$v / v_{th}$')\nax.set_ylabel('$f(v)$ (normalized / 정규화)')\nax.set_title('로그 스케일 — 고에너지 꼬리 / Log Scale — Suprathermal Tails')\nax.legend()\nax.set_xlim(0, 8)\nax.set_ylim(1e-10, 2)\nax.grid(True, alpha=0.3)\n\nplt.suptitle(\n    'Maxwellian vs Kappa 분포 (Marsch 2006, Fig. 7 스타일)\\n'\n    'Maxwellian vs Kappa Distribution (Marsch 2006, Fig. 7 style)',\n    fontsize=14, y=1.02)\nplt.tight_layout()\nplt.show()

### 1.2 멱법칙 꼬리 확인 / Power-Law Tail Verification\n\n고에너지 극한($v \\gg v_\\kappa$)에서 kappa 분포는 멱법칙 거동을 보입니다:\n\nIn the high-energy limit ($v \\gg v_\\kappa$), the kappa distribution exhibits power-law behavior:\n\n$$f_\\kappa(v) \\propto v^{-2(\\kappa+1)}$$\n\n로그-로그 그래프에서 기울기가 $-2(\\kappa+1)$임을 확인합니다.\n\nWe verify that the slope in log-log space equals $-2(\\kappa+1)$.

In [ ]:
# --- Power-law tail verification on log-log axes ---\n\nv_norm_loglog = np.logspace(0.3, 2, 500)  # v/v_th from ~2 to 100\nv_loglog = v_norm_loglog * v_th\n\nfig, ax = plt.subplots(figsize=(10, 7))\n\nfor kappa, color in zip(kappa_values, colors):\n    f_kap = kappa_3d(v_loglog, n, T, m, kappa)\n    ax.loglog(v_norm_loglog, f_kap / f_kap[0], '-', color=color,\n             lw=2, label=f'$\\\\kappa$ = {kappa}')\n\n    # Overplot theoretical power-law slope.\n    slope = -2 * (kappa + 1)\n    power_law = (v_norm_loglog / v_norm_loglog[0])**slope\n    ax.loglog(v_norm_loglog, power_law, ':', color=color, lw=1,\n             label=f'slope = $-2(\\\\kappa+1)$ = {slope}')\n\n# Maxwellian for reference (exponential decay, steeper than any power law).\nf_m = maxwellian_3d(v_loglog, n, T, m)\nax.loglog(v_norm_loglog, f_m / f_m[0], 'k-', lw=2.5, label='Maxwellian')\n\nax.set_xlabel('$v / v_{th}$ (log scale)')\nax.set_ylabel('$f(v)$ (normalized, log scale / 정규화, 로그 스케일)')\nax.set_title(\n    '멱법칙 꼬리 확인: $f(v) \\\\propto v^{-2(\\\\kappa+1)}$\\n'\n    'Power-Law Tail Verification: $f(v) \\\\propto v^{-2(\\\\kappa+1)}$')\nax.legend(ncol=2, fontsize=9)\nax.set_ylim(1e-30, 2)\nax.grid(True, alpha=0.3, which='both')\nplt.tight_layout()\nplt.show()

---\n\n## Part 2: 태양풍 전자 VDF — Core-Halo-Strahl 모델\n## Part 2: Solar Wind Electron VDF — Core-Halo-Strahl Model\n\n### 이론적 배경 / Theoretical Background\n\n태양풍 전자의 VDF는 세 성분으로 구성됩니다 (논문 Section 2.1):\n\nThe solar wind electron VDF consists of three components (paper Section 2.1):\n\n1. **Core**: 저온의 Maxwell 분포. 전체 밀도의 대부분을 차지 / Low-temperature Maxwellian. Contains most of the density\n2. **Halo**: 고온의 kappa 분포. 초열 전자 집단 / High-temperature kappa distribution. Suprathermal electron population\n3. **Strahl**: 자기장 방향을 따른 좁은 빔. 열 흐름을 운반 / Narrow beam along the magnetic field. Carries heat flux\n\n$$f_e(v_\\parallel, v_\\perp) = f_{\\text{core}}(v_\\parallel, v_\\perp) + f_{\\text{halo}}(v_\\parallel, v_\\perp) + f_{\\text{strahl}}(v_\\parallel, v_\\perp)$$\n\n**관측 파라미터 (Helios, ~0.3 AU) / Observed parameters (Helios, ~0.3 AU):**\n- Core: $n_c = 30.8$ cm$^{-3}$, $T_c = 1.6 \\times 10^5$ K\n- Halo: $n_h = 2.2$ cm$^{-3}$, $T_h = 8.9 \\times 10^5$ K\n- Strahl: 자기장 방향의 좁은 빔 / Narrow beam along B-field

In [ ]:
def bimax_2d(vpar, vperp, n, T_par, T_perp, m, v_drift=0.0):\n    \"\"\"Compute bi-Maxwellian distribution in 2D velocity space.\n\n    Args:\n        vpar: Parallel velocity array [m/s].\n        vperp: Perpendicular velocity array [m/s].\n        n: Number density [m^-3].\n        T_par: Parallel temperature [K].\n        T_perp: Perpendicular temperature [K].\n        m: Particle mass [kg].\n        v_drift: Drift speed along parallel direction [m/s].\n\n    Returns:\n        2D array of f(v_par, v_perp) [s^3/m^6] on the grid.\n    \"\"\"\n    vth_par = np.sqrt(2 * k_B * T_par / m)\n    vth_perp = np.sqrt(2 * k_B * T_perp / m)\n    Vpar, Vperp = np.meshgrid(vpar, vperp, indexing='ij')\n    f = (n / (np.pi**1.5 * vth_par * vth_perp**2)\n         * np.exp(-((Vpar - v_drift) / vth_par)**2\n                  - (Vperp / vth_perp)**2))\n    return f\n\n\ndef kappa_2d(vpar, vperp, n, T_par, T_perp, m, kappa, v_drift=0.0):\n    \"\"\"Compute 2D bi-kappa distribution function.\n\n    Args:\n        vpar: Parallel velocity array [m/s].\n        vperp: Perpendicular velocity array [m/s].\n        n: Number density [m^-3].\n        T_par: Parallel temperature [K].\n        T_perp: Perpendicular temperature [K].\n        m: Particle mass [kg].\n        kappa: Kappa index.\n        v_drift: Drift speed [m/s].\n\n    Returns:\n        2D array of f(v_par, v_perp) on the grid.\n    \"\"\"\n    vk_par = np.sqrt((2 * kappa - 3) / kappa * k_B * T_par / m)\n    vk_perp = np.sqrt((2 * kappa - 3) / kappa * k_B * T_perp / m)\n    Vpar, Vperp = np.meshgrid(vpar, vperp, indexing='ij')\n    prefactor = (n / (np.pi**1.5 * kappa**1.5 * vk_par * vk_perp**2)\n                 * gamma_func(kappa + 1) / gamma_func(kappa - 0.5))\n    f = prefactor * (1 + ((Vpar - v_drift)**2 / (kappa * vk_par**2)\n                          + Vperp**2 / (kappa * vk_perp**2)))**(-(kappa + 1))\n    return f\n\n\ndef strahl_2d(vpar, vperp, n, T_par, T_perp, m, v_drift):\n    \"\"\"Compute strahl component as a narrow drifting bi-Maxwellian beam.\n\n    Args:\n        vpar: Parallel velocity array [m/s].\n        vperp: Perpendicular velocity array [m/s].\n        n: Number density [m^-3].\n        T_par: Parallel temperature (narrow) [K].\n        T_perp: Perpendicular temperature (narrow) [K].\n        m: Particle mass [kg].\n        v_drift: Beam drift speed along B [m/s].\n\n    Returns:\n        2D array of strahl f(v_par, v_perp).\n    \"\"\"\n    return bimax_2d(vpar, vperp, n, T_par, T_perp, m, v_drift=v_drift)\n\n\nprint(\"2D VDF component functions defined.\")

In [ ]:
# --- Core-Halo-Strahl electron VDF (Figure 1 style) ---\n\n# Observed parameters from Helios at ~0.3 AU.\nn_core = 30.8e6       # Core density [m^-3]\nT_core = 1.6e5        # Core temperature [K]\nn_halo = 2.2e6        # Halo density [m^-3]\nT_halo = 8.9e5        # Halo temperature [K]\nkappa_halo = 4.0      # Halo kappa index\nn_strahl = 0.5e6      # Strahl density [m^-3]\nT_strahl_par = 3.0e5  # Strahl parallel temperature [K]\nT_strahl_perp = 1.0e5 # Strahl perpendicular temperature [K]\n\n# Thermal speed for normalization.\nv_th_core = np.sqrt(2 * k_B * T_core / m_e)\nv_strahl_drift = 4.0 * v_th_core  # Strahl drift ~4 v_th along B\n\n# Velocity grid.\nnv = 300\nv_max = 8.0 * v_th_core\nvpar = np.linspace(-v_max, v_max, nv)\nvperp = np.linspace(-v_max, v_max, nv)\n\n# Compute each component.\nf_core = bimax_2d(vpar, vperp, n_core, T_core, T_core, m_e)\nf_halo = kappa_2d(vpar, vperp, n_halo, T_halo, T_halo, m_e, kappa_halo)\nf_str = strahl_2d(vpar, vperp, n_strahl, T_strahl_par, T_strahl_perp,\n                  m_e, v_strahl_drift)\n\n# Total VDF.\nf_total = f_core + f_halo + f_str\n\n# --- 2D contour plot ---\nfig, axes = plt.subplots(1, 3, figsize=(18, 5.5))\n\n# Normalize velocities to v_th_core.\nvpar_norm = vpar / v_th_core\nvperp_norm = vperp / v_th_core\n\n# Log10 of VDF for contour levels.\nf_total_log = np.log10(np.maximum(f_total, 1e-50))\nf_core_log = np.log10(np.maximum(f_core, 1e-50))\n\nlev_min = f_total_log.max() - 8\nlev_max = f_total_log.max()\nlevels = np.linspace(lev_min, lev_max, 20)\n\n# Panel 1: Total VDF.\nax = axes[0]\ncs = ax.contour(vpar_norm, vperp_norm, f_total_log.T, levels=levels, cmap='viridis')\nax.set_xlabel('$v_\\\\parallel / v_{th,c}$')\nax.set_ylabel('$v_\\\\perp / v_{th,c}$')\nax.set_title('전체 전자 VDF / Total Electron VDF')\nax.set_aspect('equal')\nax.axhline(0, color='gray', lw=0.5)\nax.axvline(0, color='gray', lw=0.5)\nax.set_xlim(-6, 6)\nax.set_ylim(-6, 6)\n\n# Panel 2: Core only.\nax = axes[1]\nax.contour(vpar_norm, vperp_norm, f_core_log.T, levels=levels, cmap='Blues')\nax.set_xlabel('$v_\\\\parallel / v_{th,c}$')\nax.set_title('Core만 / Core Only (Maxwellian)')\nax.set_aspect('equal')\nax.axhline(0, color='gray', lw=0.5)\nax.axvline(0, color='gray', lw=0.5)\nax.set_xlim(-6, 6)\nax.set_ylim(-6, 6)\n\n# Panel 3: Halo + Strahl.\nf_hs_log = np.log10(np.maximum(f_halo + f_str, 1e-50))\nax = axes[2]\nax.contour(vpar_norm, vperp_norm, f_hs_log.T, levels=levels, cmap='Reds')\nax.set_xlabel('$v_\\\\parallel / v_{th,c}$')\nax.set_title('Halo + Strahl')\nax.set_aspect('equal')\nax.axhline(0, color='gray', lw=0.5)\nax.axvline(0, color='gray', lw=0.5)\nax.set_xlim(-6, 6)\nax.set_ylim(-6, 6)\n\nplt.suptitle(\n    '태양풍 전자 VDF: Core-Halo-Strahl 모델 (Marsch 2006, Fig. 1 스타일)\\n'\n    'Solar Wind Electron VDF: Core-Halo-Strahl Model (Marsch 2006, Fig. 1 style)',\n    fontsize=13, y=1.02)\nplt.tight_layout()\nplt.show()

### 2.1 1D 절단면 / 1D Cuts Along and Perpendicular to B\n\n자기장 방향($v_\\parallel$)과 수직 방향($v_\\perp$)의 1D 절단면에서 core, halo, strahl의 상대적 기여를 확인합니다.\n\n1D cuts along ($v_\\parallel$) and perpendicular to ($v_\\perp$) the magnetic field show the relative contributions of core, halo, and strahl.

In [ ]:
# --- 1D cuts through the electron VDF ---\n\n# Index of vperp=0 and vpar=0 in the grid.\ni_mid = nv // 2\n\nfig, axes = plt.subplots(1, 2, figsize=(16, 6))\n\n# --- Cut along B (v_perp = 0) ---\nax = axes[0]\nax.semilogy(vpar_norm, f_core[:, i_mid], 'b-', lw=2, label='Core')\nax.semilogy(vpar_norm, f_halo[:, i_mid], 'r--', lw=2, label='Halo')\nax.semilogy(vpar_norm, f_str[:, i_mid], 'g-.', lw=2, label='Strahl')\nax.semilogy(vpar_norm, f_total[:, i_mid], 'k-', lw=2.5, label='Total / 합계')\nax.set_xlabel('$v_\\\\parallel / v_{th,c}$')\nax.set_ylabel('$f(v_\\\\parallel, v_\\\\perp=0)$')\nax.set_title('B 방향 절단면 / Cut Along B ($v_\\\\perp = 0$)')\nax.legend()\nax.set_xlim(-6, 8)\nax.set_ylim(1e-5 * f_total[:, i_mid].max(), 2 * f_total[:, i_mid].max())\nax.grid(True, alpha=0.3)\n\n# --- Cut perpendicular to B (v_par = 0) ---\nax = axes[1]\nax.semilogy(vperp_norm, f_core[i_mid, :], 'b-', lw=2, label='Core')\nax.semilogy(vperp_norm, f_halo[i_mid, :], 'r--', lw=2, label='Halo')\nax.semilogy(vperp_norm, f_str[i_mid, :], 'g-.', lw=2, label='Strahl')\nax.semilogy(vperp_norm, f_total[i_mid, :], 'k-', lw=2.5, label='Total / 합계')\nax.set_xlabel('$v_\\\\perp / v_{th,c}$')\nax.set_ylabel('$f(v_\\\\parallel=0, v_\\\\perp)$')\nax.set_title('B 수직 방향 절단면 / Cut Perpendicular to B ($v_\\\\parallel = 0$)')\nax.legend()\nax.set_xlim(-6, 6)\nax.set_ylim(1e-5 * f_total[i_mid, :].max(), 2 * f_total[i_mid, :].max())\nax.grid(True, alpha=0.3)\n\nplt.suptitle(\n    '전자 VDF 1D 절단면: Core-Halo-Strahl 분리\\n'\n    'Electron VDF 1D Cuts: Core-Halo-Strahl Decomposition',\n    fontsize=13, y=1.02)\nplt.tight_layout()\nplt.show()

---\n\n## Part 3: 태양 대기의 충돌성 / Collisionality in the Solar Atmosphere\n\n### 이론적 배경 / Theoretical Background\n\n태양 대기의 충돌 조건은 채층(chromosphere)에서 태양풍까지 극적으로 변합니다 (논문 Table 1). **Knudsen 수** $\\varepsilon = \\lambda_c / L$이 핵심 파라미터입니다:\n\nCollisionality conditions change dramatically from the chromosphere to the solar wind (paper Table 1). The **Knudsen number** $\\varepsilon = \\lambda_c / L$ is the key parameter:\n\n- $\\varepsilon \\ll 1$: 충돌 지배 (유체 기술 유효) / Collision-dominated (fluid description valid)\n- $\\varepsilon \\sim 1$: 전이 영역 / Transition region\n- $\\varepsilon \\gg 1$: 비충돌 (운동론적 기술 필요) / Collisionless (kinetic description needed)\n\n**Coulomb 충돌 파라미터 / Coulomb collision parameters:**\n\n- 열 속도 / Thermal speed: $v_{th} = \\sqrt{2k_BT/m}$\n- Coulomb 로그 / Coulomb logarithm: $\\ln\\Lambda \\approx 23 - \\ln(n_e^{1/2} T_e^{-3/2})$ (for $T_e > 10$ eV)\n- 충돌 주파수 / Collision frequency: $\\nu_{ee} = \\frac{n e^4 \\ln\\Lambda}{4\\pi\\varepsilon_0^2 m_e^2 v_{th,e}^3}$\n- 평균 자유 경로 / Mean free path: $\\lambda_c = v_{th} / \\nu$

In [ ]:
def coulomb_log(n_e, T_e):\n    \"\"\"Compute Coulomb logarithm for electron-electron collisions.\n\n    Args:\n        n_e: Electron density [m^-3].\n        T_e: Electron temperature [K].\n\n    Returns:\n        ln(Lambda) dimensionless.\n    \"\"\"\n    # Standard NRL formula for T_e > ~10 eV.\n    T_eV = k_B * T_e / e_charge\n    if T_eV > 10:\n        return 24.0 - np.log(np.sqrt(n_e * 1e-6) / T_eV)\n    else:\n        return 23.0 - np.log(np.sqrt(n_e * 1e-6) * T_eV**(-1.5))\n\n\ndef collision_freq_ee(n_e, T_e):\n    \"\"\"Compute electron-electron Coulomb collision frequency.\n\n    Args:\n        n_e: Electron density [m^-3].\n        T_e: Electron temperature [K].\n\n    Returns:\n        Collision frequency [Hz].\n    \"\"\"\n    v_th = np.sqrt(2 * k_B * T_e / m_e)\n    ln_lam = coulomb_log(n_e, T_e)\n    return n_e * e_charge**4 * ln_lam / (4 * np.pi * epsilon_0**2 * m_e**2 * v_th**3)\n\n\ndef collision_freq_pp(n_p, T_p):\n    \"\"\"Compute proton-proton Coulomb collision frequency.\n\n    Args:\n        n_p: Proton density [m^-3].\n        T_p: Proton temperature [K].\n\n    Returns:\n        Collision frequency [Hz].\n    \"\"\"\n    v_th = np.sqrt(2 * k_B * T_p / m_p)\n    ln_lam = coulomb_log(n_p, T_p)\n    return n_p * e_charge**4 * ln_lam / (4 * np.pi * epsilon_0**2 * m_p**2 * v_th**3)\n\n\ndef mean_free_path(v_th, nu):\n    \"\"\"Compute mean free path.\n\n    Args:\n        v_th: Thermal speed [m/s].\n        nu: Collision frequency [Hz].\n\n    Returns:\n        Mean free path [m].\n    \"\"\"\n    return v_th / nu\n\n\n# --- Reproduce Table 1 from Marsch (2006) ---\n\n# Define regions: name, n [m^-3], T [K], L (scale height) [m]\nregions = [\n    ('Chromosphere / 채층',  1e16, 1e4,   2e6),     # n=10^10 cm^-3, T~10^4 K\n    ('Transition Region / 전이영역', 1e15, 5e5, 5e6),\n    ('Lower Corona / 하부 코로나', 1e14, 1e6, 5e7),\n    ('Upper Corona / 상부 코로나', 1e13, 2e6, 1e8),\n    ('Solar Wind 0.3 AU', 3e7, 2e5, 5e10),\n    ('Solar Wind 1 AU / 1 AU 태양풍', 1e7, 1e5, 1e11),\n]\n\nprint(f\"{'Region / 영역':<35} {'n [m^-3]':>12} {'T [K]':>10} {'v_th [km/s]':>12}\"\n      f\" {'nu_ee [Hz]':>12} {'lambda_c [m]':>14} {'L [m]':>12} {'Kn = lambda/L':>14}\")\nprint('-' * 140)\n\nkn_values = []\nregion_names = []\n\nfor name, n_e, T_e, L in regions:\n    v_th = np.sqrt(2 * k_B * T_e / m_e)\n    nu_ee = collision_freq_ee(n_e, T_e)\n    lam_c = mean_free_path(v_th, nu_ee)\n    kn = lam_c / L\n    kn_values.append(kn)\n    region_names.append(name)\n    print(f\"{name:<35} {n_e:>12.2e} {T_e:>10.1e} {v_th/1e3:>12.1f}\"\n          f\" {nu_ee:>12.2e} {lam_c:>14.2e} {L:>12.2e} {kn:>14.2e}\")

In [ ]:
# --- Visualize collisionality transition ---\n\nfig, ax = plt.subplots(figsize=(12, 6))\n\nx_pos = np.arange(len(region_names))\nbars = ax.bar(x_pos, kn_values, color=['#2196F3', '#4CAF50', '#FF9800',\n                                       '#FF5722', '#9C27B0', '#607D8B'],\n              edgecolor='black', alpha=0.8)\n\nax.set_yscale('log')\nax.set_xticks(x_pos)\nax.set_xticklabels([r.split('/')[0].strip() for r in region_names],\n                   rotation=30, ha='right', fontsize=10)\nax.set_ylabel('Knudsen Number $\\\\varepsilon = \\\\lambda_c / L$')\nax.set_title(\n    '태양 대기에서 태양풍까지의 충돌성 전이 (Marsch 2006, Table 1)\\n'\n    'Collisionality Transition from Solar Atmosphere to Solar Wind (Marsch 2006, Table 1)')\n\n# Mark the transition boundary.\nax.axhline(1.0, color='red', ls='--', lw=2, label='$\\\\varepsilon = 1$ (전이 경계 / transition)')\nax.axhspan(0, 1, alpha=0.08, color='blue',\n           label='충돌 지배 / Collisional ($\\\\varepsilon < 1$)')\nax.axhspan(1, ax.get_ylim()[1], alpha=0.08, color='red',\n           label='비충돌 / Collisionless ($\\\\varepsilon > 1$)')\n\n# Add value labels on bars.\nfor bar, kn in zip(bars, kn_values):\n    ax.text(bar.get_x() + bar.get_width()/2, kn * 1.5,\n            f'{kn:.1e}', ha='center', va='bottom', fontsize=9)\n\nax.legend(loc='upper left', fontsize=10)\nax.set_ylim(1e-10, 1e8)\nax.grid(True, alpha=0.3, which='both', axis='y')\nplt.tight_layout()\nplt.show()

---\n\n## Part 4: Exospheric 태양풍 모델 / Exospheric Solar Wind Model\n\n### 이론적 배경 / Theoretical Background\n\n**Bernoulli 에너지 방정식 (Eq. 2)**: polytropic ($\\gamma$) 코로나 팽창에서의 에너지 보존으로, 코로나 온도로부터 종단(terminal) 태양풍 속도를 결정합니다:\n\n**Bernoulli energy equation (Eq. 2)**: Energy conservation in polytropic ($\\gamma$) coronal expansion, determining the terminal solar wind speed from coronal temperature:\n\n$$\\frac{1}{2}V^2 = \\frac{\\gamma}{\\gamma-1}\\frac{2k_BT_C}{m_p} - \\frac{GM_\\odot}{R_\\odot}$$\n\n**유효 포텐셜 (Eq. 5)**: 각 입자가 느끼는 유효 포텐셜은 자기 모멘트 보존, 중력, 정전기 포텐셜의 합입니다:\n\n**Effective potential (Eq. 5)**: The effective potential felt by each particle combines magnetic moment conservation, gravity, and electrostatic potential:\n\n$$\\Psi(r) = \\mu B(r) + m\\Phi_g(r) + q\\Phi_e(r)$$\n\n**Parker spiral 자기장 (Eq. 4)**:\n\n$$B(r) = B_0 \\left(\\frac{R_\\odot}{r}\\right)^2 \\sqrt{1 + \\left(\\frac{\\Omega_\\odot r \\sin\\theta}{V_{sw}}\\right)^2}$$

In [ ]:
# --- 4.1 Bernoulli equation: Terminal wind speed vs coronal temperature ---\n\ndef terminal_wind_speed(T_C, gamma=5.0/3.0):\n    \"\"\"Compute terminal solar wind speed from Bernoulli equation (Eq. 2).\n\n    Args:\n        T_C: Coronal temperature [K].\n        gamma: Polytropic index.\n\n    Returns:\n        Terminal wind speed [m/s]. Returns 0 if energy is insufficient.\n    \"\"\"\n    # Kinetic energy per proton from thermal reservoir.\n    E_thermal = (gamma / (gamma - 1)) * (2 * k_B * T_C / m_p)\n    # Gravitational binding energy per proton.\n    E_grav = G_grav * M_sun / R_sun\n    E_net = E_thermal - E_grav\n    if isinstance(E_net, np.ndarray):\n        v = np.where(E_net > 0, np.sqrt(2 * np.maximum(E_net, 0)), 0.0)\n    else:\n        v = np.sqrt(2 * E_net) if E_net > 0 else 0.0\n    return v\n\n\n# Temperature range.\nT_C = np.linspace(0.5e6, 5e6, 500)\n\n# Compute for different polytropic indices.\ngammas = [1.1, 1.2, 5.0/3.0, 1.5]\nlabels_gamma = ['$\\\\gamma$ = 1.1 (near-isothermal / 준등온)',\n                '$\\\\gamma$ = 1.2',\n                '$\\\\gamma$ = 5/3 (adiabatic / 단열)',\n                '$\\\\gamma$ = 1.5']\ncolors_gamma = ['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e']\n\nfig, ax = plt.subplots(figsize=(10, 7))\n\nfor gam, label, color in zip(gammas, labels_gamma, colors_gamma):\n    V = terminal_wind_speed(T_C, gamma=gam) / 1e3  # km/s\n    ax.plot(T_C / 1e6, V, '-', color=color, lw=2, label=label)\n\n# Mark observed values.\nax.axhline(400, color='gray', ls=':', lw=1, alpha=0.7)\nax.text(0.6, 410, 'Slow wind ~400 km/s', fontsize=9, color='gray')\nax.axhline(750, color='gray', ls=':', lw=1, alpha=0.7)\nax.text(0.6, 760, 'Fast wind ~750 km/s', fontsize=9, color='gray')\n\n# Mark escape speed.\nv_esc = np.sqrt(2 * G_grav * M_sun / R_sun) / 1e3\nax.axhline(v_esc, color='black', ls='--', lw=1, alpha=0.5)\nax.text(0.6, v_esc + 10, f'Escape speed = {v_esc:.0f} km/s', fontsize=9)\n\nax.set_xlabel('코로나 온도 / Coronal Temperature $T_C$ [MK]')\nax.set_ylabel('종단 태양풍 속도 / Terminal Wind Speed $V$ [km/s]')\nax.set_title(\n    'Bernoulli 방정식: 코로나 온도 vs 종단 태양풍 속도 (Marsch 2006, Eq. 2)\\n'\n    'Bernoulli Equation: Coronal Temperature vs Terminal Wind Speed')\nax.legend(fontsize=10)\nax.set_xlim(0.5, 5)\nax.set_ylim(0, 1200)\nax.grid(True, alpha=0.3)\nplt.tight_layout()\nplt.show()

In [ ]:
# --- 4.2 Parker spiral magnetic field and effective potential ---\n\n# Solar rotation rate.\nOmega_sun = 2.87e-6  # Solar angular velocity [rad/s] (~27 day period)\ntheta = np.pi / 4     # Heliographic latitude (45 deg)\n\ndef parker_spiral_B(r, B0, V_sw, theta=np.pi/2):\n    \"\"\"Compute Parker spiral magnetic field magnitude (Eq. 4).\n\n    Args:\n        r: Heliocentric distance array [m].\n        B0: Surface radial field strength [T].\n        V_sw: Solar wind speed [m/s].\n        theta: Colatitude [rad].\n\n    Returns:\n        B(r) in [T].\n    \"\"\"\n    B_r = B0 * (R_sun / r)**2\n    spiral_term = (Omega_sun * r * np.sin(theta) / V_sw)**2\n    return B_r * np.sqrt(1 + spiral_term)\n\n\ndef gravitational_potential(r):\n    \"\"\"Compute gravitational potential per unit mass.\n\n    Args:\n        r: Heliocentric distance [m].\n\n    Returns:\n        Phi_g [J/kg].\n    \"\"\"\n    return -G_grav * M_sun / r\n\n\ndef electrostatic_potential_pannekoek(r, T_e=1e6):\n    \"\"\"Compute Pannekoek-Rosseland electrostatic potential (Eq. 3 style).\n\n    In an electron-proton plasma, the ambipolar electric field creates\n    a potential: Phi_e(r) ~ (m_p/2e) * GM_sun/r.\n\n    Args:\n        r: Heliocentric distance [m].\n        T_e: Electron temperature [K] (for thermal reference).\n\n    Returns:\n        Phi_e in [V].\n    \"\"\"\n    # Pannekoek-Rosseland: e*Phi_e = -(m_p - m_e)/(2) * GM/r\n    # For proton-electron: Phi_e ~ -m_p*G*M / (2*e*r)\n    return -(m_p - m_e) * G_grav * M_sun / (2 * e_charge * r)\n\n\n# --- Plot B(r) and potentials ---\nr = np.linspace(1.01 * R_sun, 1.5 * AU, 2000)\nr_Rs = r / R_sun\n\nfig, axes = plt.subplots(2, 2, figsize=(14, 10))\n\n# Panel 1: Parker spiral B field.\nax = axes[0, 0]\nfor V_sw, color, label in [(400e3, 'blue', 'Slow wind 400 km/s'),\n                            (750e3, 'red', 'Fast wind 750 km/s')]:\n    B_equator = parker_spiral_B(r, 2e-4, V_sw, theta=np.pi/2)\n    B_pole = parker_spiral_B(r, 2e-4, V_sw, theta=np.pi/6)\n    ax.semilogy(r / AU, B_equator * 1e9, '-', color=color, label=f'{label} (equator)')\n    ax.semilogy(r / AU, B_pole * 1e9, '--', color=color, label=f'{label} (30$^\\\\circ$)')\nax.set_xlabel('Heliocentric Distance [AU]')\nax.set_ylabel('$B$ [nT]')\nax.set_title('Parker Spiral 자기장 / Parker Spiral Magnetic Field')\nax.legend(fontsize=9)\nax.grid(True, alpha=0.3)\nax.set_ylim(1e-1, 1e6)\n\n# Panel 2: Gravitational potential.\nax = axes[0, 1]\nPhi_g = gravitational_potential(r)\nax.plot(r / R_sun, Phi_g / 1e6, 'k-', lw=2)\nax.set_xlabel('$r / R_\\\\odot$')\nax.set_ylabel('$\\\\Phi_g$ [MJ/kg]')\nax.set_title('중력 포텐셜 / Gravitational Potential')\nax.set_xlim(1, 50)\nax.grid(True, alpha=0.3)\n\n# Panel 3: Electrostatic potential (Figure 6 style).\nax = axes[1, 0]\nr_close = np.linspace(1.01 * R_sun, 30 * R_sun, 1000)\nPhi_e = electrostatic_potential_pannekoek(r_close)\nax.plot(r_close / R_sun, Phi_e, 'r-', lw=2)\nax.set_xlabel('$r / R_\\\\odot$')\nax.set_ylabel('$\\\\Phi_e$ [V]')\nax.set_title('Pannekoek-Rosseland 정전기 포텐셜\\nElectrostatic Potential (Fig. 6 style)')\nax.grid(True, alpha=0.3)\n\n# Panel 4: Effective potential for proton.\nax = axes[1, 1]\nfor T_e, color in [(1e6, 'blue'), (2e6, 'red')]:\n    # Effective potential: gravity + electrostatic.\n    Psi_grav = m_p * gravitational_potential(r_close)\n    Psi_elec = e_charge * electrostatic_potential_pannekoek(r_close, T_e)\n    # Magnetic mirror term (assume B ~ 1/r^2).\n    B_r = 2e-4 * (R_sun / r_close)**2\n    mu_mag = 0.5 * m_p * (100e3)**2 / B_r[0]  # Typical magnetic moment.\n    Psi_mag = mu_mag * B_r\n    Psi_total = Psi_grav + Psi_elec + Psi_mag\n    ax.plot(r_close / R_sun, Psi_total / e_charge,\n            '-', color=color, lw=2, label=f'$T_e$ = {T_e/1e6:.0f} MK')\nax.set_xlabel('$r / R_\\\\odot$')\nax.set_ylabel('$\\\\Psi_{eff}$ / $e$ [V]')\nax.set_title('양성자 유효 포텐셜 / Proton Effective Potential (Eq. 5)')\nax.legend()\nax.grid(True, alpha=0.3)\n\nplt.suptitle(\n    'Exospheric 모델 구성요소 (Marsch 2006, Eqs. 2-5)\\n'\n    'Exospheric Model Components (Marsch 2006, Eqs. 2-5)',\n    fontsize=14, y=1.02)\nplt.tight_layout()\nplt.show()

---\n\n## Part 5: 분산 관계 — Alfven-Cyclotron 파동 / Dispersion Relations — Alfven-Cyclotron Waves\n\n### 이론적 배경 / Theoretical Background\n\n차가운 플라즈마(cold plasma)에서 자기장 방향으로 전파하는 전자기파의 분산 관계는 이온과 전자의 cyclotron 공명에 의해 결정됩니다:\n\nThe dispersion relation for electromagnetic waves propagating parallel to the magnetic field in a cold plasma is determined by ion and electron cyclotron resonances:\n\n$$\\frac{k^2 c^2}{\\omega^2} = 1 - \\frac{\\omega_{pi}^2}{\\omega(\\omega \\mp \\Omega_i)} - \\frac{\\omega_{pe}^2}{\\omega(\\omega \\pm \\Omega_e)}$$\n\n여기서 상단 부호(-)는 좌편광(L-mode, ion-cyclotron), 하단 부호(+)는 우편광(R-mode, whistler)입니다.\n\nwhere upper sign (-) is left-hand polarized (L-mode, ion-cyclotron) and lower sign (+) is right-hand polarized (R-mode, whistler).\n\n**핵심 주파수 / Key frequencies:**\n- Proton cyclotron: $\\Omega_i = eB/m_p$\n- Electron cyclotron: $\\Omega_e = eB/m_e$\n- Proton plasma: $\\omega_{pi} = \\sqrt{n_i e^2 / (\\varepsilon_0 m_p)}$\n- Alfven speed: $V_A = B/\\sqrt{\\mu_0 n_i m_p}$\n\n**Cyclotron 공명 조건 (Eq. 7 근처) / Cyclotron resonance condition:**\n\n$$w_\\parallel = \\frac{\\omega'(k_\\parallel) \\pm \\Omega_j}{k_\\parallel}$$

In [ ]:
# --- Cold plasma parallel dispersion relation ---\n\ndef cold_plasma_dispersion_parallel(omega, k, B0, n0, mode='L'):\n    \"\"\"Compute the cold plasma dispersion function for parallel propagation.\n\n    Returns D(omega, k) = 0 at the dispersion relation roots.\n\n    Args:\n        omega: Angular frequency [rad/s].\n        k: Parallel wavenumber [rad/m].\n        B0: Background magnetic field [T].\n        n0: Plasma density [m^-3].\n        mode: 'L' for left-hand (ion-cyclotron) or 'R' for right-hand (whistler).\n\n    Returns:\n        Dispersion function value (0 when on dispersion curve).\n    \"\"\"\n    Omega_i = e_charge * B0 / m_p\n    Omega_e = e_charge * B0 / m_e\n    omega_pi2 = n0 * e_charge**2 / (epsilon_0 * m_p)\n    omega_pe2 = n0 * e_charge**2 / (epsilon_0 * m_e)\n\n    if mode == 'L':  # Left-hand (ion-cyclotron branch)\n        D = (k**2 * c_light**2 / omega**2\n             - 1\n             + omega_pi2 / (omega * (omega - Omega_i))\n             + omega_pe2 / (omega * (omega + Omega_e)))\n    else:  # Right-hand (whistler branch)\n        D = (k**2 * c_light**2 / omega**2\n             - 1\n             + omega_pi2 / (omega * (omega + Omega_i))\n             + omega_pe2 / (omega * (omega - Omega_e)))\n    return D\n\n\ndef solve_dispersion(k_array, B0, n0, mode='L', omega_guess=None):\n    \"\"\"Solve cold plasma dispersion relation for omega(k).\n\n    Args:\n        k_array: Array of wavenumbers [rad/m].\n        B0: Magnetic field [T].\n        n0: Density [m^-3].\n        mode: 'L' or 'R'.\n        omega_guess: Initial guess for first k value.\n\n    Returns:\n        Array of angular frequencies [rad/s].\n    \"\"\"\n    Omega_i = e_charge * B0 / m_p\n    V_A = B0 / np.sqrt(mu_0 * n0 * m_p)\n\n    omega_arr = np.zeros_like(k_array)\n\n    for i, k in enumerate(k_array):\n        if i == 0:\n            # Start from Alfven wave: omega = k * V_A\n            guess = k * V_A if omega_guess is None else omega_guess\n        else:\n            # Extrapolate from previous solutions.\n            if i == 1:\n                guess = omega_arr[i-1] * k / k_array[i-1]\n            else:\n                guess = omega_arr[i-1] + (omega_arr[i-1] - omega_arr[i-2])\n\n        # Use bracketing solver.\n        if mode == 'L':\n            omega_max = 0.999 * Omega_i  # Cannot exceed cyclotron frequency.\n        else:\n            omega_max = 0.999 * e_charge * B0 / m_e\n\n        try:\n            func = lambda w: cold_plasma_dispersion_parallel(w, k, B0, n0, mode)\n            # Search in reasonable range.\n            w_lo = 1e-3 * Omega_i\n            w_hi = min(guess * 3, omega_max)\n            if w_hi <= w_lo:\n                w_hi = omega_max\n            omega_arr[i] = brentq(func, w_lo, w_hi, maxiter=200)\n        except (ValueError, RuntimeError):\n            omega_arr[i] = np.nan\n\n    return omega_arr\n\n\n# Plasma parameters: typical corona.\nB0 = 5e-4         # 5 Gauss = 5e-4 T\nn0 = 1e14         # 10^8 cm^-3\n\nOmega_i = e_charge * B0 / m_p\nOmega_e = e_charge * B0 / m_e\nV_A = B0 / np.sqrt(mu_0 * n0 * m_p)\nd_i = c_light / np.sqrt(n0 * e_charge**2 / (epsilon_0 * m_p))  # Ion inertial length.\n\nprint(f\"Proton cyclotron frequency: Omega_i = {Omega_i:.2e} rad/s\")\nprint(f\"  f_ci = {Omega_i/(2*np.pi):.2f} Hz\")\nprint(f\"Alfven speed: V_A = {V_A/1e3:.1f} km/s\")\nprint(f\"Ion inertial length: d_i = {d_i:.2f} m\")\nprint(f\"V_A / c = {V_A/c_light:.2e}\")

In [ ]:
# --- Plot dispersion relations: omega vs k ---\n\n# Use normalized units: k*d_i vs omega/Omega_i.\nk_norm = np.linspace(0.01, 5.0, 300)\nk_arr = k_norm / d_i  # Physical k values.\n\n# Solve L-mode (ion-cyclotron) branch.\nomega_L = solve_dispersion(k_arr, B0, n0, mode='L')\n\n# For R-mode, solve in the low-frequency (below Omega_i) Alfvenic range.\n# The R-mode doesn't have a resonance at Omega_i, it passes through.\n# We'll compute the MHD-like fast magnetosonic / whistler branch.\n# Simplified: use the Alfven/magnetosonic approximation.\n# For parallel propagation: R-mode goes through Omega_i and up to Omega_e.\n# We use the simplified dispersion: omega^2 = k^2 V_A^2 / (1 - omega/Omega_i) for L\n# and omega^2 = k^2 V_A^2 / (1 + omega/Omega_i) for R (below Omega_i regime).\n\n# Simplified Hall-MHD dispersion (valid for omega << Omega_e):\n# L-mode: omega = k*V_A * sqrt(1/(1 - omega/Omega_i))\n# R-mode: omega = k*V_A * sqrt(1/(1 + omega/Omega_i))\ndef hall_mhd_dispersion(k, V_A, Omega_i, mode='L'):\n    \"\"\"Solve simplified Hall-MHD dispersion iteratively.\n\n    Args:\n        k: Wavenumber array.\n        V_A: Alfven speed.\n        Omega_i: Ion cyclotron frequency.\n        mode: 'L' or 'R'.\n\n    Returns:\n        omega array.\n    \"\"\"\n    omega = k * V_A  # Initial guess: Alfven wave.\n    for _ in range(100):  # Iterate to convergence.\n        if mode == 'L':\n            omega_new = k * V_A / np.sqrt(1 - omega / Omega_i)\n        else:\n            omega_new = k * V_A / np.sqrt(1 + omega / Omega_i)\n        # Damp oscillations.\n        omega = 0.5 * omega + 0.5 * omega_new\n        # Clamp L-mode below Omega_i.\n        if mode == 'L':\n            omega = np.minimum(omega, 0.999 * Omega_i)\n    return omega\n\nomega_L_hall = hall_mhd_dispersion(k_arr, V_A, Omega_i, mode='L')\nomega_R_hall = hall_mhd_dispersion(k_arr, V_A, Omega_i, mode='R')\n\nfig, axes = plt.subplots(1, 2, figsize=(16, 6))\n\n# --- Panel 1: omega/Omega_i vs k*d_i ---\nax = axes[0]\nax.plot(k_norm, omega_L_hall / Omega_i, 'b-', lw=2,\n        label='L-mode (ion-cyclotron / 이온 사이클로트론)')\nax.plot(k_norm, omega_R_hall / Omega_i, 'r-', lw=2,\n        label='R-mode (whistler / 휘슬러)')\nax.plot(k_norm, k_norm * V_A * d_i / (Omega_i * d_i), 'k--', lw=1,\n        alpha=0.5, label='Alfven: $\\\\omega = k V_A$')\nax.axhline(1.0, color='blue', ls=':', lw=1, alpha=0.5)\nax.text(0.1, 1.02, '$\\\\Omega_i$', color='blue', fontsize=11)\n\nax.set_xlabel('$k d_i$')\nax.set_ylabel('$\\\\omega / \\\\Omega_i$')\nax.set_title('분산 관계 (선형 스케일)\\nDispersion Relations (Linear Scale)')\nax.legend(fontsize=10)\nax.set_xlim(0, 5)\nax.set_ylim(0, 3)\nax.grid(True, alpha=0.3)\n\n# --- Panel 2: Phase speed / V_A ---\nax = axes[1]\nvph_L = omega_L_hall / k_arr / V_A\nvph_R = omega_R_hall / k_arr / V_A\nax.plot(k_norm, vph_L, 'b-', lw=2, label='L-mode')\nax.plot(k_norm, vph_R, 'r-', lw=2, label='R-mode')\nax.axhline(1.0, color='k', ls='--', lw=1, alpha=0.5, label='$V_A$')\n\n# Mark dissipation wavenumber scaling (Eq. 7): k_d ~ Omega_i / V_A.\nk_d = Omega_i / V_A\nk_d_norm = k_d * d_i\nax.axvline(k_d_norm, color='green', ls='-.', lw=2, alpha=0.7,\n           label=f'$k_d d_i$ = {k_d_norm:.2f} (dissipation scale / 소산 스케일)')\n\nax.set_xlabel('$k d_i$')\nax.set_ylabel('$v_{ph} / V_A$')\nax.set_title('위상 속도 / Phase Speed')\nax.legend(fontsize=9)\nax.set_xlim(0, 5)\nax.set_ylim(0, 4)\nax.grid(True, alpha=0.3)\n\nplt.suptitle(\n    '차가운 플라즈마 분산 관계: 평행 전파 (Marsch 2006, Section 5)\\n'\n    'Cold Plasma Dispersion: Parallel Propagation (Marsch 2006, Section 5)',\n    fontsize=13, y=1.02)\nplt.tight_layout()\nplt.show()

---\n\n## Part 6: Cyclotron 공명과 Pitch-Angle 확산 / Cyclotron Resonance and Pitch-Angle Diffusion\n\n### 이론적 배경 / Theoretical Background\n\nCyclotron 공명은 입자의 자이로 운동 주파수와 파동의 Doppler-shifted 주파수가 일치할 때 발생합니다. 속도 공간에서 공명 조건은 원(circle) 또는 타원을 형성합니다:\n\nCyclotron resonance occurs when the particle's gyration frequency matches the Doppler-shifted wave frequency. In velocity space, the resonance condition forms circles or ellipses:\n\n$$\\omega - k_\\parallel v_\\parallel = \\pm \\Omega_j$$\n\n이를 풀면 $v_\\parallel$에 대한 공명 속도를 얻습니다:\n\nSolving for the resonant parallel velocity:\n\n$$v_{\\parallel,res} = \\frac{\\omega \\mp \\Omega_j}{k_\\parallel}$$\n\n**준선형 확산 (Eq. 56 스타일)**: 파동-입자 상호작용에 의해 VDF는 공명 원을 따라 확산되며, 최종적으로 **확산 고원(diffusion plateau)**을 형성합니다 (Figure 14 스타일).\n\n**Quasilinear diffusion (Eq. 56 style)**: Wave-particle interactions cause the VDF to diffuse along resonance circles, eventually forming a **diffusion plateau** (Figure 14 style).\n\n확산은 속도 공간에서 특성 곡선(characteristic curves)을 따라 일어나며, 이 곡선은 $(v_\\parallel - v_{ph})^2 + v_\\perp^2 = \\text{const}$ 형태의 원입니다.\n\nDiffusion occurs along characteristic curves in velocity space, which are circles of the form $(v_\\parallel - v_{ph})^2 + v_\\perp^2 = \\text{const}$.

In [ ]:
# --- 6.1 Cyclotron resonance circles in velocity space ---\n\n# Normalized units: velocities in V_A, frequencies in Omega_i.\n# Resonance condition: v_par_res = (omega - n*Omega_i) / k_par\n# where n = +1 (normal cyclotron), n = -1 (anomalous).\n\n# Use dispersion solutions from Part 5 to get (omega, k) pairs.\n# Select a few representative wave modes.\nk_test = np.array([0.5, 1.0, 2.0, 3.0]) / d_i  # wavenumbers\nomega_test_L = hall_mhd_dispersion(k_test, V_A, Omega_i, mode='L')\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 6))\n\n# --- Panel 1: Resonance circles for L-mode waves ---\nax = axes[0]\ncolors_res = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']\n\nfor i, (k, omega) in enumerate(zip(k_test, omega_test_L)):\n    # Resonance center: v_par = omega/k (wave phase speed).\n    v_ph = omega / k\n    # Resonant parallel velocity: v_par_res = (omega - Omega_i) / k.\n    v_res = (omega - Omega_i) / k\n\n    # Diffusion circle centered at (v_ph, 0) in velocity space.\n    # Radius determined by the resonant particle's total speed.\n    # Circle: (v_par - v_ph)^2 + v_perp^2 = R^2\n    # At resonance: R^2 = (v_res - v_ph)^2 + v_perp_res^2 for some v_perp.\n    R = abs(v_res - v_ph)\n    # Draw circles with different radii.\n    theta_circ = np.linspace(0, 2 * np.pi, 200)\n    for R_mult in [1.0, 1.5, 2.0]:\n        R_actual = R * R_mult\n        v_par_circ = v_ph + R_actual * np.cos(theta_circ)\n        v_perp_circ = R_actual * np.sin(theta_circ)\n        if R_mult == 1.0:\n            ax.plot(v_par_circ / V_A, v_perp_circ / V_A, '-',\n                    color=colors_res[i], lw=1.5,\n                    label=f'$k d_i$ = {k*d_i:.1f}, $\\\\omega/\\\\Omega_i$ = {omega/Omega_i:.2f}')\n        else:\n            ax.plot(v_par_circ / V_A, v_perp_circ / V_A, '-',\n                    color=colors_res[i], lw=0.7, alpha=0.5)\n\n    # Mark resonant v_parallel.\n    ax.axvline(v_res / V_A, color=colors_res[i], ls=':', lw=0.8, alpha=0.5)\n\nax.set_xlabel('$v_\\\\parallel / V_A$')\nax.set_ylabel('$v_\\\\perp / V_A$')\nax.set_title('L-mode Cyclotron 공명 원\\nL-mode Cyclotron Resonance Circles')\nax.set_xlim(-5, 2)\nax.set_ylim(-3, 3)\nax.set_aspect('equal')\nax.legend(fontsize=8, loc='lower left')\nax.grid(True, alpha=0.3)\nax.axhline(0, color='gray', lw=0.5)\nax.axvline(0, color='gray', lw=0.5)\n\n# --- Panel 2: Diffusion plateau formation (Figure 14 style) ---\nax = axes[1]\n\n# Create initial bi-Maxwellian proton VDF with T_perp > T_par.\nnv2 = 200\nv_max2 = 4 * V_A\nvpar2 = np.linspace(-v_max2, v_max2, nv2)\nvperp2 = np.linspace(-v_max2, v_max2, nv2)\n\n# Anisotropic proton VDF.\nT_par_p = 5e5   # K\nT_perp_p = 15e5  # K (T_perp/T_par = 3)\nn_p = n0\nf_proton = bimax_2d(vpar2, vperp2, n_p, T_par_p, T_perp_p, m_p)\n\n# Simulate diffusion plateau by flattening VDF along resonance circles.\n# We'll modify the VDF to show plateau formation.\nf_plateau = f_proton.copy()\nVpar2, Vperp2 = np.meshgrid(vpar2, vperp2, indexing='ij')\n\n# Apply plateau in the resonance region (v_par < -V_A).\n# Flatten f along circles centered at v_ph ~ V_A.\nv_ph_plateau = V_A\nR_grid = np.sqrt((Vpar2 - v_ph_plateau)**2 + Vperp2**2)\n\n# In the region where resonance operates, make f constant along circles.\nmask_resonance = (Vpar2 < -0.5 * V_A) & (R_grid < 3.5 * V_A)\nfor R_val in np.linspace(0, 3.5 * V_A, 100):\n    ring = (np.abs(R_grid - R_val) < 0.15 * V_A) & mask_resonance\n    if ring.any():\n        f_plateau[ring] = np.mean(f_plateau[ring])\n\nf_plateau_log = np.log10(np.maximum(f_plateau, 1e-50))\nf_proton_log = np.log10(np.maximum(f_proton, 1e-50))\nlev_min_p = f_proton_log.max() - 6\nlevels_p = np.linspace(lev_min_p, f_proton_log.max(), 15)\n\n# Plot modified VDF with plateau.\nax.contour(vpar2 / V_A, vperp2 / V_A, f_plateau_log.T,\n           levels=levels_p, cmap='inferno')\nax.contour(vpar2 / V_A, vperp2 / V_A, f_proton_log.T,\n           levels=levels_p, colors='gray', linewidths=0.5, alpha=0.4)\n\n# Overlay a few resonance circles.\nfor R_show in [1.5 * V_A, 2.0 * V_A, 2.5 * V_A]:\n    ax.plot((v_ph_plateau + R_show * np.cos(theta_circ)) / V_A,\n            R_show * np.sin(theta_circ) / V_A,\n            'w--', lw=0.8, alpha=0.6)\n\nax.set_xlabel('$v_\\\\parallel / V_A$')\nax.set_ylabel('$v_\\\\perp / V_A$')\nax.set_title('확산 고원 형성 (Fig. 14 스타일)\\nDiffusion Plateau Formation (Fig. 14 style)')\nax.set_xlim(-4, 4)\nax.set_ylim(-3, 3)\nax.set_aspect('equal')\nax.axhline(0, color='white', lw=0.5, alpha=0.3)\nax.axvline(0, color='white', lw=0.5, alpha=0.3)\n\nplt.suptitle(\n    'Cyclotron 공명과 준선형 확산 (Marsch 2006, Section 6)\\n'\n    'Cyclotron Resonance and Quasilinear Diffusion (Marsch 2006, Section 6)',\n    fontsize=13, y=1.02)\nplt.tight_layout()\nplt.show()

---\n\n## Part 7: 온도 비등방성 조절 / Temperature Anisotropy Regulation\n\n### 이론적 배경 / Theoretical Background\n\n태양풍 양성자의 온도 비등방성($T_\\perp / T_\\parallel$)은 두 가지 instability에 의해 조절됩니다 (논문 Eq. 57, Figure 17):\n\nThe temperature anisotropy ($T_\\perp / T_\\parallel$) of solar wind protons is regulated by two instabilities (paper Eq. 57, Figure 17):\n\n**1. Ion-cyclotron instability** ($T_\\perp > T_\\parallel$, 상한 / upper bound):\n$$\\frac{T_\\perp}{T_\\parallel} - 1 = \\frac{S_p}{\\beta_\\parallel^{\\alpha_p}}$$\n- $S_p \\approx 0.64$, $\\alpha_p \\approx 0.41$ (Hellinger et al., 2006 피팅 / fit)\n\n**2. Parallel firehose instability** ($T_\\perp < T_\\parallel$, 하한 / lower bound):\n$$\\frac{T_\\perp}{T_\\parallel} - 1 = -\\frac{S_f}{\\beta_\\parallel^{\\alpha_f}}$$\n- $S_f \\approx 1.4$, $\\alpha_f \\approx 0.55$\n\nHelios 관측 데이터에서 태양풍 양성자 데이터 포인트가 이 두 경계 사이에 분포하며, 태양풍이 한계 안정성(marginal stability) 근처에서 유지됨을 보여줍니다.\n\nHelios observations show that solar wind proton data points are distributed between these two boundaries, demonstrating that the solar wind maintains near marginal stability.

In [ ]:
# --- 7.1 Anisotropy-beta instability thresholds (Eq. 57, Fig. 17 style) ---\n\ndef cyclotron_instability_threshold(beta_par, S=0.64, alpha=0.41):\n    \"\"\"Compute ion-cyclotron instability threshold (upper bound on T_perp/T_par).\n\n    Args:\n        beta_par: Parallel plasma beta array.\n        S: Fit parameter.\n        alpha: Fit parameter.\n\n    Returns:\n        T_perp/T_par ratio at threshold.\n    \"\"\"\n    return 1 + S / beta_par**alpha\n\n\ndef firehose_instability_threshold(beta_par, S=1.4, alpha=0.55):\n    \"\"\"Compute parallel firehose instability threshold (lower bound).\n\n    Args:\n        beta_par: Parallel plasma beta array.\n        S: Fit parameter.\n        alpha: Fit parameter.\n\n    Returns:\n        T_perp/T_par ratio at threshold.\n    \"\"\"\n    return 1 - S / beta_par**alpha\n\n\ndef mirror_instability_threshold(beta_par, S=0.87, alpha=0.56):\n    \"\"\"Compute mirror instability threshold (alternative upper bound).\n\n    Args:\n        beta_par: Parallel plasma beta array.\n        S: Fit parameter.\n        alpha: Fit parameter.\n\n    Returns:\n        T_perp/T_par ratio at threshold.\n    \"\"\"\n    return 1 + S / beta_par**alpha\n\n\n# Beta range.\nbeta_par = np.logspace(-1.5, 1.5, 500)\n\n# Compute thresholds.\nR_cyclotron = cyclotron_instability_threshold(beta_par)\nR_firehose = firehose_instability_threshold(beta_par)\nR_mirror = mirror_instability_threshold(beta_par, S=0.87, alpha=0.56)\n\n# --- Generate synthetic data mimicking Helios observations ---\nnp.random.seed(42)\nn_points = 5000\n\n# Beta values from log-uniform distribution.\nbeta_synth = 10**np.random.uniform(-1, 1.2, n_points)\n\n# Anisotropy values: constrained between thresholds.\nR_upper = cyclotron_instability_threshold(beta_synth)\nR_lower = firehose_instability_threshold(beta_synth)\nR_lower = np.maximum(R_lower, 0.1)  # Physical lower limit.\n\n# Generate anisotropy: peaked near 1 with scatter toward boundaries.\nR_synth = np.random.lognormal(mean=0, sigma=0.25, size=n_points)\n\n# Clip to stay between instability boundaries.\nR_synth = np.clip(R_synth, R_lower, R_upper)\n\n# Add more points near the cyclotron boundary (observed clustering).\nn_boundary = 1000\nbeta_boundary = 10**np.random.uniform(-0.5, 1.0, n_boundary)\nR_boundary = cyclotron_instability_threshold(beta_boundary) * np.random.uniform(0.7, 1.0, n_boundary)\nR_boundary = np.maximum(R_boundary, firehose_instability_threshold(beta_boundary))\n\nbeta_all = np.concatenate([beta_synth, beta_boundary])\nR_all = np.concatenate([R_synth, R_boundary])\n\n# --- Plot ---\nfig, ax = plt.subplots(figsize=(10, 8))\n\n# Scatter plot of synthetic data.\nax.scatter(beta_all, R_all, s=1, c='gray', alpha=0.3, rasterized=True,\n           label='모의 데이터 / Synthetic Data (Helios-like)')\n\n# 2D histogram for density.\nhist_range = [[np.log10(beta_all.min()), np.log10(beta_all.max())],\n              [0, np.max(R_all)]]\nH, xedges, yedges = np.histogram2d(\n    np.log10(beta_all), R_all, bins=60, range=[[-1.5, 1.5], [0, 4]])\nax.contourf(10**((xedges[:-1] + xedges[1:])/2),\n            (yedges[:-1] + yedges[1:])/2,\n            H.T, levels=8, cmap='YlOrRd', alpha=0.6)\n\n# Instability boundaries.\nax.plot(beta_par, R_cyclotron, 'b-', lw=2.5,\n        label='Ion-cyclotron 불안정성 / Ion-cyclotron instability')\nax.plot(beta_par, R_mirror, 'g--', lw=2,\n        label='Mirror 불안정성 / Mirror instability')\nax.plot(beta_par, R_firehose, 'r-', lw=2.5,\n        label='Firehose 불안정성 / Firehose instability')\nax.axhline(1.0, color='k', ls=':', lw=1, alpha=0.5)\nax.text(0.12, 1.02, '$T_\\\\perp = T_\\\\parallel$ (등방 / isotropic)', fontsize=9)\n\nax.set_xscale('log')\nax.set_xlabel('$\\\\beta_{\\\\parallel p}$')\nax.set_ylabel('$T_{\\\\perp p} / T_{\\\\parallel p}$')\nax.set_title(\n    '양성자 온도 비등방성 vs 평행 Beta (Marsch 2006, Fig. 17 스타일)\\n'\n    'Proton Temperature Anisotropy vs Parallel Beta (Marsch 2006, Fig. 17 style)')\nax.legend(loc='upper right', fontsize=10)\nax.set_xlim(0.03, 30)\nax.set_ylim(0, 5)\nax.grid(True, alpha=0.3, which='both')\n\n# Annotate regions.\nax.annotate('안정 영역\\nStable Region',\n            xy=(1, 1.5), fontsize=12, ha='center',\n            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))\nax.annotate('Cyclotron\\n불안정 / Unstable',\n            xy=(0.2, 4), fontsize=10, ha='center', color='blue')\nax.annotate('Firehose\\n불안정 / Unstable',\n            xy=(5, 0.2), fontsize=10, ha='center', color='red')\n\nplt.tight_layout()\nplt.show()

---\n\n## Part 8: 현대적 맥락 — Marsch에서 Parker Solar Probe까지\n## Part 8: Modern Context — From Marsch to Parker Solar Probe\n\n### 8.1 개념과 현대 관측의 연결 / Connecting Concepts to Modern Measurements\n\nMarsch (2006)의 리뷰는 Helios와 Ulysses 시대의 지식을 종합한 것입니다. 그 이후 Parker Solar Probe (PSP, 2018-)와 Solar Orbiter (2020-)가 코로나에 전례 없이 가까이 접근하여 이 논문의 예측과 모델을 직접 검증하고 있습니다.\n\nMarsch (2006) synthesized knowledge from the Helios and Ulysses era. Since then, Parker Solar Probe (PSP, 2018-) and Solar Orbiter (2020-) have approached the corona at unprecedented distances, directly testing this paper's predictions and models.\n\n| Marsch (2006) 개념 / Concept | 현대 관측 결과 / Modern Findings |\n|---|---|\n| **비Maxwell VDF** / Non-Maxwellian VDFs | PSP가 0.17 AU에서 양성자 빔과 강한 $T_\\perp > T_\\parallel$ 확인. VDF 구조가 태양에 가까울수록 더 비평형적 / PSP confirmed proton beams and strong $T_\\perp > T_\\parallel$ at 0.17 AU. VDFs more non-equilibrium closer to Sun |\n| **Core-halo-strahl 전자** / Core-halo-strahl electrons | Strahl 폭이 태양에 가까울수록 좁아짐 확인. Halo가 strahl 산란으로 형성되는 증거 / Strahl width narrows closer to Sun. Evidence that halo forms from strahl scattering |\n| **Cyclotron 공명 가열** / Cyclotron resonance heating | PSP가 cyclotron-scale 파동과 이온 가열 간의 상관관계 직접 관측 / PSP directly observed correlation between cyclotron-scale waves and ion heating |\n| **온도 비등방성 조절** / Temperature anisotropy regulation | PSP 데이터도 cyclotron-firehose 경계 내에 분포. 0.17 AU에서도 한계 안정성 유지 / PSP data also bounded by cyclotron-firehose limits. Marginal stability maintained at 0.17 AU |\n| **Exospheric 모델** / Exospheric models | Kappa 분포 기반 모델이 태양풍 가속을 일부 설명하나, 파동 가열이 필수적임을 PSP가 확인 / Kappa-based models partially explain acceleration, but PSP confirmed wave heating is essential |\n| **파동-입자 상호작용** / Wave-particle interactions | PSP의 *in situ* 자기장 요동 측정으로 Alfven-cyclotron 파동의 에너지 cascade 직접 관측 / PSP's *in situ* magnetic fluctuation measurements directly observed Alfven-cyclotron wave energy cascade |\n| **충돌성 전이** / Collisionality transition | PSP가 코로나 근처에서 비충돌-충돌 전이 영역을 처음으로 직접 통과 / PSP first direct traversal of collisionless-collisional transition near corona |

In [ ]:
# --- 8.2 Timeline of kinetic solar wind physics development ---\n\ntimeline = [\n    (1958, 'Parker', '유체역학적 태양풍 모델 / Hydrodynamic solar wind model'),\n    (1966, 'Hundhausen et al.', '태양풍 이온 관측 시작 / First solar wind ion observations'),\n    (1974, 'Helios 1 발사 / launch', '0.3 AU까지 in situ 측정 시작 / In situ to 0.3 AU begins'),\n    (1979, 'Scudder & Olbert', '초기 exospheric 모델 / Early exospheric model'),\n    (1982, 'Marsch et al.', '양성자 VDF 비등방성 발견 / Proton VDF anisotropy discovered'),\n    (1990, 'Ulysses 발사 / launch', '고위도 태양풍 관측 / High-latitude solar wind'),\n    (1995, 'SOHO 발사 / launch', '코로나 원격 관측 혁명 / Coronal remote sensing revolution'),\n    (2003, 'Lamy, Zouganelis', '현대 kappa 기반 exospheric 모델 / Modern kappa exospheric models'),\n    (2006, 'Marsch (이 논문 / this paper)', '운동론의 포괄적 종합 / Comprehensive kinetic synthesis'),\n    (2018, 'Parker Solar Probe', '코로나 직접 탐사 시작 / Direct coronal exploration begins'),\n    (2020, 'Solar Orbiter', '내부 태양권 다중 관점 관측 / Multi-viewpoint inner heliosphere'),\n    (2021, 'PSP Encounter 8-10', '처음으로 Alfven 표면 통과 / First crossing of Alfven surface'),\n    (2024, 'PSP Encounter 17+', '~9.8 Rs 근접 통과 / Closest approach ~9.8 Rs'),\n]\n\nfig, ax = plt.subplots(figsize=(14, 8))\n\nyears = [t[0] for t in timeline]\nlabels = [f\"{t[1]}\\n{t[2]}\" for t in timeline]\n\n# Plot timeline.\nax.scatter(years, range(len(years)), s=100, c='steelblue', zorder=3)\nax.vlines(years, -0.5, [i for i in range(len(years))],\n          color='steelblue', alpha=0.3, lw=1)\n\nfor i, (year, name, desc) in enumerate(timeline):\n    # Alternate label positions for readability.\n    ha = 'left' if i % 2 == 0 else 'right'\n    offset = 1.5 if i % 2 == 0 else -1.5\n    ax.annotate(f\"{year}: {name}\",\n                xy=(year, i), xytext=(year + offset, i),\n                fontsize=9, fontweight='bold',\n                ha=ha, va='center',\n                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))\n    ax.text(year + offset * 1.5, i - 0.3, desc, fontsize=7,\n            ha=ha, va='top', color='gray', style='italic')\n\n# Highlight Marsch 2006.\nidx_marsch = 8\nax.scatter([2006], [idx_marsch], s=200, c='red', zorder=4, marker='*')\n\nax.set_xlabel('연도 / Year')\nax.set_yticks([])\nax.set_title(\n    '운동론적 태양풍 물리학 발전 타임라인\\n'\n    'Timeline of Kinetic Solar Wind Physics Development')\nax.set_xlim(1955, 2027)\nax.grid(True, alpha=0.2, axis='x')\nplt.tight_layout()\nplt.show()

In [ ]:
# --- 8.3 Compare Marsch predictions with PSP observations ---\n# Schematic comparison of key parameters at different heliocentric distances.\n\nfig, axes = plt.subplots(1, 3, figsize=(16, 5))\n\n# Heliocentric distances [AU].\ndist_au = np.array([0.05, 0.1, 0.17, 0.3, 0.5, 1.0])\ndist_labels = ['0.05\\n(PSP\\nmin)', '0.1', '0.17\\n(PSP)', '0.3\\n(Helios\\nmin)',\n               '0.5', '1.0\\n(Earth)']\n\n# Panel 1: Proton temperature anisotropy T_perp/T_par.\nax = axes[0]\n# Marsch/Helios data: T_perp/T_par increases closer to Sun in fast wind.\nR_aniso_helios = np.array([np.nan, np.nan, np.nan, 2.5, 1.8, 1.2])\nR_aniso_psp = np.array([3.5, 3.0, 2.8, np.nan, np.nan, np.nan])\nax.plot(dist_au[3:], R_aniso_helios[3:], 'bs-', ms=8, lw=2,\n        label='Helios (Marsch 2006)')\nax.plot(dist_au[:3], R_aniso_psp[:3], 'r^-', ms=8, lw=2,\n        label='PSP (2018-)')\nax.set_xlabel('Heliocentric Distance [AU]')\nax.set_ylabel('$T_{\\\\perp p} / T_{\\\\parallel p}$')\nax.set_title('양성자 비등방성 / Proton Anisotropy')\nax.legend(fontsize=9)\nax.set_xlim(0, 1.1)\nax.grid(True, alpha=0.3)\nax.invert_xaxis()\n\n# Panel 2: Proton beam drift speed / V_A.\nax = axes[1]\nbeam_helios = np.array([np.nan, np.nan, np.nan, 1.2, 1.0, 0.7])\nbeam_psp = np.array([1.5, 1.4, 1.3, np.nan, np.nan, np.nan])\nax.plot(dist_au[3:], beam_helios[3:], 'bs-', ms=8, lw=2, label='Helios')\nax.plot(dist_au[:3], beam_psp[:3], 'r^-', ms=8, lw=2, label='PSP')\nax.axhline(1.0, color='gray', ls=':', lw=1)\nax.text(0.5, 1.02, '$V_A$ (Alfven speed)', fontsize=8, color='gray')\nax.set_xlabel('Heliocentric Distance [AU]')\nax.set_ylabel('$v_{beam} / V_A$')\nax.set_title('양성자 빔 드리프트 속도 / Proton Beam Drift')\nax.legend(fontsize=9)\nax.set_xlim(0, 1.1)\nax.grid(True, alpha=0.3)\nax.invert_xaxis()\n\n# Panel 3: Electron strahl width.\nax = axes[2]\nstrahl_helios = np.array([np.nan, np.nan, np.nan, 15, 20, 35])\nstrahl_psp = np.array([5, 8, 10, np.nan, np.nan, np.nan])\nax.plot(dist_au[3:], strahl_helios[3:], 'bs-', ms=8, lw=2, label='Helios')\nax.plot(dist_au[:3], strahl_psp[:3], 'r^-', ms=8, lw=2, label='PSP')\nax.set_xlabel('Heliocentric Distance [AU]')\nax.set_ylabel('Strahl Width [degrees / 도]')\nax.set_title('전자 Strahl 폭 / Electron Strahl Width')\nax.legend(fontsize=9)\nax.set_xlim(0, 1.1)\nax.grid(True, alpha=0.3)\nax.invert_xaxis()\n\nplt.suptitle(\n    'Marsch (2006) 예측 vs Parker Solar Probe 관측 (개략적 비교)\\n'\n    'Marsch (2006) Predictions vs Parker Solar Probe Observations (Schematic)',\n    fontsize=13, y=1.02)\nplt.tight_layout()\nplt.show()

---\n\n## 핵심 요점 / Key Takeaways\n\n1. **VDF의 비평형성이 핵심 / Non-equilibrium VDFs are central**: 태양풍 입자의 속도 분포는 Maxwell 분포에서 크게 벗어나며, kappa 분포가 관측적 VDF를 잘 기술합니다. 이는 MHD를 넘어서는 운동론적 기술의 필요성을 보여줍니다.\n   Solar wind particle VDFs deviate strongly from Maxwellian, and kappa distributions describe observed VDFs well. This demonstrates the need for kinetic descriptions beyond MHD.\n\n2. **충돌성의 급격한 변화 / Dramatic collisionality change**: 채층(Knudsen $\\ll 1$)에서 태양풍(Knudsen $\\gg 1$)까지 충돌 조건이 급격히 변하며, 유체 기술에서 운동론적 기술로의 전환이 필요합니다.\n   Collisionality changes dramatically from chromosphere ($\\text{Kn} \\ll 1$) to solar wind ($\\text{Kn} \\gg 1$), requiring a transition from fluid to kinetic descriptions.\n\n3. **파동-입자 상호작용이 코로나 가열의 핵심 / Wave-particle interactions key to coronal heating**: Cyclotron 공명에 의한 pitch-angle 확산이 이온을 가열하고 VDF의 비등방성을 생성합니다.\n   Cyclotron resonance-driven pitch-angle diffusion heats ions and creates VDF anisotropy.\n\n4. **자연이 한계 안정성을 유지 / Nature maintains marginal stability**: 온도 비등방성이 cyclotron과 firehose instability 경계 사이에 항상 유지되며, Parker Solar Probe가 이를 0.17 AU에서도 확인했습니다.\n   Temperature anisotropy is always maintained between cyclotron and firehose instability boundaries, confirmed by PSP at 0.17 AU.\n\n---\n\n## 참고문헌 / References\n\n- Marsch, E. (2006), \"Kinetic Physics of the Solar Corona and Solar Wind\", *Living Rev. Solar Phys.*, 3, 1. [DOI: 10.12942/lrsp-2006-1](https://doi.org/10.12942/lrsp-2006-1)\n- Parker, E. N. (1958), \"Dynamics of the Interplanetary Gas and Magnetic Fields\", *ApJ*, 128, 664.\n- Maksimovic, M., Pierrard, V., & Lemaire, J. F. (1997), \"A kinetic model of the solar wind with Kappa distribution functions in the corona\", *A&A*, 324, 725.\n- Hellinger, P. et al. (2006), \"Solar wind proton temperature anisotropy: Linear theory and WIND/SWE observations\", *GRL*, 33, L09101.\n- Kasper, J. C. et al. (2021), \"Parker Solar Probe enters the magnetically dominated solar corona\", *PRL*, 127, 255101.\n- Verniero, J. L. et al. (2020), \"Parker Solar Probe observations of proton beams simultaneous with ion-scale waves\", *ApJS*, 248, 5.